## 准备数据

In [21]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [22]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [23]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.truncated_normal([784, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))

        self.W2 = tf.Variable(tf.random.truncated_normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 784])              # [batch, 28, 28] -> [batch, 784]
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [24]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [26]:
train_data, test_data = mnist_dataset()
for epoch in range(100):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 1.786038 ; accuracy 0.45986667
epoch 1 : loss 1.7760524 ; accuracy 0.46665
epoch 2 : loss 1.7661651 ; accuracy 0.47356668
epoch 3 : loss 1.7563753 ; accuracy 0.4804
epoch 4 : loss 1.7466812 ; accuracy 0.48686665
epoch 5 : loss 1.7370821 ; accuracy 0.49331668
epoch 6 : loss 1.7275769 ; accuracy 0.49986666
epoch 7 : loss 1.7181643 ; accuracy 0.5056
epoch 8 : loss 1.7088431 ; accuracy 0.5114167
epoch 9 : loss 1.6996121 ; accuracy 0.5177
epoch 10 : loss 1.6904709 ; accuracy 0.52321666
epoch 11 : loss 1.6814182 ; accuracy 0.5283667
epoch 12 : loss 1.6724532 ; accuracy 0.5333833
epoch 13 : loss 1.663575 ; accuracy 0.5383667
epoch 14 : loss 1.6547834 ; accuracy 0.54323334
epoch 15 : loss 1.6460768 ; accuracy 0.5479
epoch 16 : loss 1.6374546 ; accuracy 0.55296665
epoch 17 : loss 1.6289167 ; accuracy 0.5578833
epoch 18 : loss 1.6204615 ; accuracy 0.56195
epoch 19 : loss 1.6120878 ; accuracy 0.56591666
epoch 20 : loss 1.6037954 ; accuracy 0.56953335
epoch 21 : loss 1.5955828 ; acc